# Day 100: Capstone: From Data to a Defensible Causal Estimate

*Beginner to Ninja: Math for Causal Inference*

The final day puts the whole course to work on one question: did a community health-worker programme lower blood pressure, by how much, for whom, and should it be targeted? We go from a causal graph to identification, from naive comparisons to doubly robust machine-learning estimates, then check overlap and balance, quantify sensitivity to hidden confounding, explore heterogeneity and design a targeting rule, and finish with a report a ministry can act on.

**Learning outcomes**

- Translate a policy question into an estimand, a causal graph and identifying assumptions.
- Estimate the effect with regression, weighting and cross-fitted AIPW, and diagnose overlap and balance.
- Quantify robustness to unmeasured confounding and test for heterogeneous effects.
- Turn the estimates into a targeting rule and a transparent, honest report.

## Theory

### The workflow

1. **Question and estimand.** Whose outcome, which contrast, which population: ATE (scale up to everyone), ATT (value for current participants) or a policy value (targeted roll-out).
2. **Causal graph and identification.** Draw the DAG; find an adjustment set (back-door criterion, Day 52) or a design (IV, RD, DiD, synthetic control) that identifies the estimand. State the assumptions in words.
3. **Estimation.** Start simple (difference in means, regression), then use estimators robust to model misspecification: weighting, matching, AIPW/DML with cross-fitted learners (Days 81–91).
4. **Diagnostics.** Overlap (propensity distribution), covariate balance (SMDs after weighting), nuisance fit quality, stability across learners and splits.
5. **Sensitivity.** How strong would hidden confounding need to be (Day 95)? Placebo outcomes and pre-trends where available.
6. **Heterogeneity and policy.** Test for heterogeneity on held-out data (BLP/GATES), then evaluate targeting rules with doubly robust welfare estimates (Days 92–94); adjust for multiplicity (Day 99).
7. **Reporting.** Estimate, uncertainty, assumptions, diagnostics, sensitivity and limitations, in language the decision maker understands.

### The case

A district offers home visits by community health workers to adults with hypertension. Enrolment was voluntary and more common among people with higher baseline blood pressure, in rural areas and with lower income. The outcome is systolic blood pressure (SBP, mmHg) one year later. Covariates: age, baseline SBP, rural residence, log income.

The DAG has each covariate pointing to both enrolment $D$ and the outcome $Y$, and $D \to Y$. With no other common causes, $\{\text{age, baseline SBP, rural, income}\}$ satisfies the back-door criterion, so under overlap

$$
\text{ATE} = E\big[E[Y \mid D = 1, X] - E[Y \mid D = 0, X]\big].
$$

### Choosing the final estimator

AIPW with cross-fitted flexible outcome models and a propensity model is consistent if either nuisance is well estimated, √n-normal when both converge reasonably fast, and comes with an influence-function SE (Days 87–90). Simpler estimators remain useful as diagnostics: large disagreements signal misspecification or overlap problems.

## Part 1 · Warm-up

### Exercise 1 · Choosing the estimand

The ministry asks: "If we offered home visits to every adult with hypertension, how much would average SBP fall?" Which estimand is this?

<details><summary>Show solution</summary>

**Answer:** the ATE over the hypertensive population (assuming everyone offered would take part; otherwise, the intention-to-treat effect of the offer).

</details>

*Your work:*

### Exercise 2 · Why the naive comparison fails

Participants' SBP after one year is only 0.5 mmHg lower than non-participants'. Name the confounder most responsible, and its direction.

<details><summary>Show solution</summary>

**Step 1.** Baseline SBP: people with high baseline SBP enrol more often and also have high follow-up SBP.

**Answer:** it biases the naive comparison upwards (towards no effect), masking the reduction.

</details>

*Your work:*

### Exercise 3 · Reading balance

Before weighting, standardised mean differences are 0.33 (age), 0.46 (baseline SBP), 0.36 (rural), $-0.27$ (log income); after inverse-propensity weighting all are below 0.02 in absolute value. Interpret.

<details><summary>Show solution</summary>

**Step 1.** Before: large imbalances (above the usual 0.1 threshold).

**Answer:** after weighting, the groups are comparable on observed covariates; the propensity model is doing its job.

</details>

*Your work:*

### Exercise 4 · Overlap

Cross-fitted propensity scores range from 0.07 to 0.98. Is overlap adequate for the ATE?

<details><summary>Show solution</summary>

**Step 1.** No score reaches 0 or 1, so every covariate profile has some chance of each status.

**Step 2.** Scores near 0.98 imply control weights up to $1/0.02 = 50$: check how many there are, and compare clipped and unclipped estimates.

</details>

*Your work:*

### Exercise 5 · A placebo check

Suggest a placebo outcome for this study.

<details><summary>Show solution</summary>

**Answer:** SBP measured *before* enrolment (e.g. two years earlier, not used as a covariate): the programme cannot affect it, so an "effect" signals residual confounding.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · Double robustness

Show that the AIPW estimand $E[\hat\mu_1 - \hat\mu_0 + D(Y - \hat\mu_1)/\hat e - (1 - D)(Y - \hat\mu_0)/(1 - \hat e)]$ equals the ATE if either the outcome models or the propensity model is correct.

<details><summary>Show solution</summary>

**Step 1.** By Day 89, Exercise 9, the treated part has bias $E[(e - \hat e)(\mu_1 - \hat\mu_1)/\hat e]$, and the control part a similar product.

**Step 2.** If $\hat\mu_d = \mu_d$, the second factor is zero; if $\hat e = e$, the first is zero. $\blacksquare$

</details>

*Your work:*

### Exercise 7 · Normalised (Hájek) weights

Show the Hájek IPW estimator $\sum_{D = 1}w_iY_i/\sum_{D = 1}w_i - \sum_{D = 0}w_iY_i/\sum_{D = 0}w_i$ is unchanged if all treated weights are multiplied by a constant, and explain why this helps.

<details><summary>Show solution</summary>

**Step 1.** A constant multiplies numerator and denominator alike within each group.

**Step 2.** So the estimator only uses relative weights; errors in the overall level of $\hat e$ do not matter, and the estimate stays within the range of observed outcomes. $\blacksquare$

</details>

*Your work:*

### Exercise 8 · What OLS estimates under heterogeneity

With a binary treatment, heterogeneous effects and a correctly specified $E[Y \mid D = 0, X]$ that is linear in $X$, show that OLS of $Y$ on $D$ and $X$ estimates $E[w(X)\tau(X)]/E[w(X)]$ with $w(X) = \mathrm{Var}(D \mid X)$ when $E[D \mid X]$ is also linear in $X$.

<details><summary>Show solution</summary>

**Step 1.** By FWL, OLS regresses $Y$ on $\tilde D = D - \text{L}(D \mid X)$, which here equals $D - e(X)$.

**Step 2.** As in Day 91, Exercise 6, the coefficient is $E[\tau\tilde D^2]/E[\tilde D^2] = E[e(1 - e)\tau]/E[e(1 - e)]$. $\blacksquare$ OLS is a variance-weighted average, not the ATE, and it is further biased when the linear specification is wrong.

</details>

*Your work:*

### Exercise 9 · An influence-function confidence interval

With AIPW scores $\hat\varphi_i$, show that $\bar{\hat\varphi} \pm 1.96\,\mathrm{sd}(\hat\varphi)/\sqrt n$ has asymptotic coverage 95%, assuming cross-fitting and nuisance-error products of order $o_p(n^{-1/2})$.

<details><summary>Show solution</summary>

**Step 1.** By orthogonality and cross-fitting (Days 89–90), $\bar{\hat\varphi} - \text{ATE} = \bar\varphi - \text{ATE} + o_p(n^{-1/2})$, with $\varphi$ the score at the true nuisances.

**Step 2.** By the central limit theorem, $\sqrt n(\bar\varphi - \text{ATE}) \to N(0, \mathrm{Var}(\varphi))$, and $\mathrm{sd}(\hat\varphi) \to \mathrm{sd}(\varphi)$.

**Step 3.** Slutsky's theorem gives the coverage. $\blacksquare$

</details>

*Your work:*

### Exercise 10 · From effects to targeting

Using Day 94, show that the rule "treat if $\tau(x) < -c$" (a reduction larger than $c$ mmHg, the cost-equivalent benefit) maximises welfare $E[\pi(X)(-\tau(X) - c)]$.

<details><summary>Show solution</summary>

**Step 1.** Pointwise, $\pi(x)(-\tau(x) - c) \le \max\{0, -\tau(x) - c\}$.

**Step 2.** The rule attains this bound at every $x$. $\blacksquare$

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · The main estimate

In the simulated district data (4,000 adults, 52% enrolled; true ATE $-5.05$ mmHg), compare the naive difference, OLS with linear controls, Hájek IPW (logistic propensity) and cross-fitted AIPW (logistic propensity, random-forest outcome models).

<details><summary>Show solution</summary>

**Step 1.** Naive: $-0.48$, badly confounded by baseline SBP.

**Step 2.** OLS: $-4.73$; IPW: $-4.65$.

**Step 3.** AIPW: $-4.88$ (SE 0.28), 95% CI $[-5.43, -4.34]$, covering the truth.

**Answer:** home visits lower SBP by about 5 mmHg on average; the three adjusted estimators agree closely, and the naive comparison would have missed the effect almost entirely.

</details>

*Your work:*

### Exercise 12 · How robust is it?

OLS gives $t = -16.3$ with about 3,994 degrees of freedom. Compute the robustness value and interpret.

<details><summary>Show solution</summary>

**Step 1.** $f = 16.3/\sqrt{3994} = 0.258$.

**Step 2.** $RV = \tfrac12(\sqrt{f^4 + 4f^2} - f^2) = 0.227$.

**Answer:** an unmeasured confounder would have to explain 23% of the residual variation in both enrolment and follow-up SBP to explain the effect away; benchmark this against baseline SBP, the strongest observed covariate.

</details>

*Your work:*

### Exercise 13 · Who benefits most?

A linear DR-learner (regressing AIPW scores on standardised covariates) gives coefficients $-2.67$ on baseline SBP (per 12 mmHg) and $+1.13$ on age (per 10 years). Interpret.

<details><summary>Show solution</summary>

**Step 1.** Each 12 mmHg higher baseline SBP adds a further 2.7 mmHg reduction.

**Step 2.** Each 10 years of age reduces the benefit by about 1.1 mmHg.

**Answer:** the programme helps most those with the highest blood pressure, and somewhat less the oldest; confirm with held-out BLP/GATES tests before acting.

</details>

*Your work:*

### Exercise 14 · A targeting rule

Valuing the programme's cost at a 4 mmHg reduction per participant, EWM over rules "visit if baseline SBP $\ge t$" picks $t = 146$ mmHg. Its true net benefit is 1.79 mmHg-equivalents per adult, versus 1.05 for visiting everyone and 1.87 for the first-best rule. Recommend.

<details><summary>Show solution</summary>

**Step 1.** Targeting raises net benefit by 70% over universal visits and achieves 96% of the first-best.

**Step 2.** The rule is simple and uses a measurement clinics already take.

**Answer:** offer visits to adults with baseline SBP of 146 mmHg or more, and evaluate the targeted roll-out.

</details>

*Your work:*

### Exercise 15 · The report

Write a four-sentence summary for the ministry.

<details><summary>Show solution</summary>

**Step 1.** "Comparing similar adults, home visits by community health workers lowered systolic blood pressure by about 5 mmHg after one year (95% CI 4.3 to 5.4)."

**Step 2.** "This relies on our having measured the main factors that drive both enrolment and blood pressure; an unmeasured factor would have to be very strong (explaining over a fifth of the remaining variation in both) to overturn the result."

**Step 3.** "Benefits are largest for people with higher blood pressure, and offering visits to adults with baseline readings of 146 mmHg or more would deliver about 70% more net benefit than offering them to everyone."

**Step 4.** "We recommend a targeted roll-out, with a randomised phase-in across districts to confirm the effect."

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(100)
n = 4000
age = rng.uniform(40, 80, n); bp0 = rng.normal(150, 12, n); rural = rng.binomial(1, .4, n); inc = rng.lognormal(0, .5, n)
X = np.column_stack([(age - 60) / 10, (bp0 - 150) / 12, rural, np.log(inc)])
e = 1 / (1 + np.exp(-(-.3 + .5 * X[:, 1] + .8 * X[:, 2] - .6 * X[:, 3] + .3 * X[:, 0]))); D = rng.binomial(1, e)
tau = -5 - 3 * X[:, 1] + X[:, 0]                                      # effect on SBP (mmHg)
mu0 = 140 + 8 * X[:, 1] + 3 * np.sin(2 * X[:, 0]) + 4 * X[:, 2] - 2 * X[:, 3] + 2 * X[:, 1] ** 2
Y = mu0 + tau * D + rng.normal(0, 8, n)
print(f"true ATE {tau.mean():.2f}   enrolled {D.mean():.2f}   naive {Y[D == 1].mean() - Y[D == 0].mean():.2f}")

Z = np.column_stack([np.ones(n), D, X]); b = np.linalg.lstsq(Z, Y, rcond=None)[0]
r = Y - Z @ b; df = n - Z.shape[1]; t = b[1] / np.sqrt(r @ r / df * np.linalg.inv(Z.T @ Z)[1, 1])
print(f"OLS {b[1]:.2f} (t = {t:.1f})")

eh, m0, m1 = np.empty(n), np.empty(n), np.empty(n)
R = lambda: RandomForestRegressor(300, min_samples_leaf=10, n_jobs=-1, random_state=0)
for tr, te in KFold(5, shuffle=True, random_state=0).split(X):
    eh[te] = LogisticRegression(C=1e6, max_iter=1000).fit(X[tr], D[tr]).predict_proba(X[te])[:, 1]
    t1, t0 = tr[D[tr] == 1], tr[D[tr] == 0]
    m1[te], m0[te] = R().fit(X[t1], Y[t1]).predict(X[te]), R().fit(X[t0], Y[t0]).predict(X[te])
w = np.where(D == 1, 1 / eh, 1 / (1 - eh))
smd = lambda w: [(np.average(X[D == 1, j], weights=w[D == 1]) - np.average(X[D == 0, j], weights=w[D == 0]))
                 / np.sqrt((X[D == 1, j].var() + X[D == 0, j].var()) / 2) for j in range(4)]
print(f"propensity range [{eh.min():.3f}, {eh.max():.3f}]   SMD raw {np.round(smd(np.ones(n)), 2)}   weighted {np.round(smd(w), 3)}")
ipw = np.average(Y[D == 1], weights=w[D == 1]) - np.average(Y[D == 0], weights=w[D == 0])
phi = m1 - m0 + D * (Y - m1) / eh - (1 - D) * (Y - m0) / (1 - eh); se = phi.std() / np.sqrt(n)
print(f"IPW {ipw:.2f}   AIPW {phi.mean():.2f} (SE {se:.2f}, CI [{phi.mean() - 1.96 * se:.2f}, {phi.mean() + 1.96 * se:.2f}])")

f = abs(t) / np.sqrt(df); print(f"robustness value {.5 * (np.sqrt(f ** 4 + 4 * f ** 2) - f ** 2):.3f}")
print("DR-linear CATE (const, age, bp0, rural, log inc):", np.linalg.lstsq(np.column_stack([np.ones(n), X]), phi, rcond=None)[0].round(2))

c = 4; best = max((np.mean((X[:, 1] >= s) * (-phi - c)), s) for s in np.linspace(-2, 2, 81))
rule = X[:, 1] >= best[1]
print(f"rule: visit if baseline SBP >= {150 + 12 * best[1]:.0f}; true welfare {np.mean(rule * (-tau - c)):.2f}, "
      f"visit all {np.mean(-tau - c):.2f}, first-best {np.mean(np.maximum(-tau - c, 0)):.2f}")

## Key takeaways

- A defensible causal estimate starts with a clear estimand and an explicit identification argument, not with a method.
- Use several estimators: agreement among regression, weighting and doubly robust ML builds confidence; disagreement is a diagnostic.
- Always report overlap, balance, sensitivity to hidden confounding and heterogeneity, and translate results into decisions with honest uncertainty.
- Congratulations: you have gone from limits and derivatives to double machine learning and policy learning. The math is now a tool you can use.